In [1]:
import requests
from bs4 import BeautifulSoup
import re

In [2]:

def get_dsf_param(url):
    headers={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'}
    with requests.session() as session:
        r=session.get(url, headers=headers)
        cookie=session.cookies.get_dict()
    soup=BeautifulSoup(r.content, 'html.parser')
    __EVENTVALIDATION=soup.find('input', {'id':'__EVENTVALIDATION'})['value']
    __VIEWSTATE=soup.find('input', {'id':'__VIEWSTATE'})['value']
    __VIEWSTATEGENERATOR=soup.find('input', {'id':'__VIEWSTATEGENERATOR'})['value']
    param=[__EVENTVALIDATION, __VIEWSTATE, __VIEWSTATEGENERATOR]
    dict_entity={re.findall("'(.*?)'", entity.get('onclick'))[1]: re.findall("'(.*?)'", entity.get('onclick'))[0]
        for entity in soup.find_all('td', {'onclick': re.compile('selectDept')})}
    return cookie, param, dict_entity

In [3]:
def get_file_value(entity, param, dict_entity, url): # p_value
        headers={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36'}
        __EVENTVALIDATION, __VIEWSTATE, __VIEWSTATEGENERATOR=param
        data={'__EVENTVALIDATION':__EVENTVALIDATION, 
                '__VIEWSTATE':__VIEWSTATE, 
                '__VIEWSTATEGENERATOR':__VIEWSTATEGENERATOR, 
                'ctl00$CPH1$deptCode':dict_entity.get(entity), 
                'ctl00$CPH1$deptDesc':entity
                }
        r=requests.post(url, headers=headers, data=data)
        soup=BeautifulSoup(r.content, 'html.parser')
        if 'Acrescimo' in url: # for 權責發生制
                file_list=[
                {'entity': re.findall("'(.*?)'", button.get('onclick'))[0], 
                'date': re.findall("'(.*?)'", button.get('onclick'))[2],
                'files': re.findall("'(.*?)'", button.get('onclick'))[1].split('||')}
                
                for button in soup.find_all('td', {'onclick': re.compile('PopDetail')})
                ]
        else: # for 現金收付制
                file_list=[
                {'entity': re.findall("'(.*?)'", button.get('onclick'))[1].split('__')[0], 
                'date': button.get_text(),
                'files': [re.findall("'(.*?)'", button.get('onclick'))[1].split('__')[1] + '__' +re.findall("'(.*?)'", button.get('onclick'))[1].split('__')[2]]
                }
                for button in soup.find_all('td', {'onclick': re.compile('javascript:assignFileAct')})
                ]
        return file_list

In [4]:
def construct_p(entity, file):
    return f'{entity}__{file.replace('.', '__')}'

In [5]:

url='https://eserv4.dsf.gov.mo/financialReport/RegimeCaixa.aspx?lang=zh'
entity='新聞局'


# url='https://eserv4.dsf.gov.mo/financialReport/RegimeAcrescimo.aspx?lang=zh'
# entity='澳門基金會'

cookie, param, dict_entity=get_dsf_param(url)
file_list=get_file_value(entity, param, dict_entity, url)

In [6]:
p_list=[]
for item in file_list:
    entity=item.get('entity')
    date=item.get('date')
    for file in item.get('files'):
        p_list.append((date, construct_p(entity, file)))

In [7]:
def download_dsf_pdf(p_value, date, cookie, url):
    download_url = 'https://eserv4.dsf.gov.mo/financialReport/downloadFile.ashx'
    # data = {'p': '90700100__90700100202519A21D6C617020B14__pdf'}  # Use the `p` value from the header
    data={'p': p_value}
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
        'Accept': 'application/pdf,application/octet-stream',
        'Content-Type': 'application/x-www-form-urlencoded',
        # 'Referer': 'https://eserv4.dsf.gov.mo/financialReport/RegimeAcrescimo.aspx?lang=zh',
        'Referer': url
    }

    # Create a session to maintain cookies
    session = requests.Session()

    # Visit the main page to establish a session
    session.get('https://eserv4.dsf.gov.mo', headers=headers)

    params = data
    r = session.get(download_url, headers=headers, params=params, cookies=cookie)
    # return r
    if r.content:
        filename = date+'_'+r.headers.get('Content-Disposition', '').split('filename=')[-1].strip('"')
        with open(filename, 'wb') as f:
            f.write(r.content)
        print(f"PDF saved as {filename}")
    else:
        print('No file found!')

In [8]:
date=p_list[1][0]
p_value=p_list[1][1]
# download_dsf_pdf(p_value, date, cookie, url)

In [9]:
download_url = 'https://eserv4.dsf.gov.mo/financialReport/downloadFile.ashx'
# data = {'p': '90700100__90700100202519A21D6C617020B14__pdf'}  # Use the `p` value from the header
data={'p': p_value}
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'application/pdf,application/octet-stream',
    'Content-Type': 'application/x-www-form-urlencoded',
    # 'Referer': 'https://eserv4.dsf.gov.mo/financialReport/RegimeAcrescimo.aspx?lang=zh',
    'Referer': url
}

# Create a session to maintain cookies
session = requests.Session()

# Visit the main page to establish a session
session.get('https://eserv4.dsf.gov.mo', headers=headers)

params = data
r = session.get(download_url, headers=headers, params=params, cookies=cookie)

In [31]:
import io
from tabula.io import read_pdf
df_list = read_pdf(io.BytesIO(r.content), pages="all", multiple_tables=True)

